# Reconciliation loop timing across E01, E02, and E03

Analyze the dedicated 50-run series for each environment. Each accepted
run has the expected reconciliation output and all 14 tasks succeeded on
their first attempt. The exception log supplies counts for interrupted,
failed, and retried runs.

Run this notebook from its directory or set PILOT_RESULTS_DIR to the
absolute path of the results data directory.

In [ ]:
import json
import os
from collections import Counter
from pathlib import Path
from random import Random
from statistics import mean, median, quantiles

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

data_dir = Path(os.environ.get("PILOT_RESULTS_DIR", "../../docs/data"))
environments = ("E01", "E02", "E03")


def read_jsonl(name):
    return [
        json.loads(line)
        for line in (data_dir / name).read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]


samples = read_jsonl("reconciliation-50-samples.jsonl")
exceptions = read_jsonl("reconciliation-50-exceptions.jsonl")
assert Counter(row["environment"] for row in samples) == dict.fromkeys(environments, 50)
assert len({row["run_id"] for row in samples + exceptions}) == len(samples) + len(exceptions)
assert all(row["environment"] in environments for row in exceptions)
assert all(
    row["result"] == {"remaining_history": [15, 7, 0], "processed": 23}
    and len(row["task_attempts"]) == 14
    and set(row["task_attempts"].values()) == {1}
    and row["duration_seconds"] >= 0
    for row in samples
)
duration = {
    env: [row["duration_seconds"] for row in samples if row["environment"] == env]
    for env in environments
}

In [ ]:
rng = Random(20261126)
excluded_counts = Counter(row["environment"] for row in exceptions)
stats = []
for env, values in duration.items():
    q1, _, q3 = quantiles(values, n=4, method="inclusive")
    bootstrap_means = [mean(rng.choices(values, k=len(values))) for _ in range(10_000)]
    percentiles = quantiles(bootstrap_means, n=1_000, method="inclusive")
    stats.append(
        {
            "Environment": env,
            "Accepted": len(values),
            "Attempts": len(values) + excluded_counts[env],
            "Min (s)": min(values),
            "Q1 (s)": q1,
            "Median (s)": median(values),
            "Q3 (s)": q3,
            "Mean (s)": mean(values),
            "Mean CI low (s)": percentiles[24],
            "Mean CI high (s)": percentiles[974],
            "Max (s)": max(values),
        }
    )
summary = pd.DataFrame(stats).set_index("Environment")
display(summary.round(3))

In [ ]:
colors = ("#125c9e", "#b54c0c", "#16805e")
fig, ax = plt.subplots(figsize=(8.5, 5.5))
fig.subplots_adjust(bottom=0.32)
for index, (env, color) in enumerate(zip(environments, colors, strict=True), start=1):
    values = duration[env]
    ax.boxplot(
        values,
        positions=[index],
        widths=0.38,
        whis=(0, 100),
        patch_artist=True,
        boxprops={"facecolor": color, "alpha": 0.16, "edgecolor": color},
        capprops={"color": color, "alpha": 0.55},
        whiskerprops={"color": color, "alpha": 0.55},
        medianprops={"color": color, "linewidth": 2.1},
    )
    scatter_x = [index + rng.uniform(-0.14, 0.14) for _ in values]
    ax.scatter(scatter_x, values, s=15, color=color, alpha=0.34, zorder=3)
    ax.scatter(index, mean(values), marker="D", s=52, color=color, edgecolors="white", zorder=4)
ax.set_xticks(range(1, 4), environments)
ax.set_ylabel("DAG duration (s)")
ax.set_title("Reconciliation loop duration · 50 accepted runs per environment")
ax.set_ylim(bottom=0)
ax.grid(axis="y", color="#e2e8f0")
fig.text(
    0.08,
    0.22,
    "Box Q1 to Q3  ·  Horizontal line Median  ·  Diamond Mean  ·  Whiskers Min / Max",
    fontsize=9,
    color="#435467",
)
fig.text(
    0.08,
    0.12,
    "Infrastructure resources differ. E01 and E02 share a Docker host. "
    "E03 uses external Kubernetes.\n"
    "Times reflect each environment's resources, configuration, and scheduling.",
    fontsize=9,
    color="#435467",
)